In [1]:
import pandas as pd

# Load bronze data
orders = pd.read_csv("/home/jovyan/work/output/bronze/orders.csv")
customers = pd.read_csv("/home/jovyan/work/output/bronze/customers.csv")
order_items = pd.read_csv("/home/jovyan/work/output/bronze/order_items.csv")
payments = pd.read_csv("/home/jovyan/work/output/bronze/payments.csv")
products = pd.read_csv("/home/jovyan/work/output/bronze/products.csv")
sellers = pd.read_csv("/home/jovyan/work/output/bronze/sellers.csv")

print("All bronze files loaded")

All bronze files loaded


In [2]:
print("orders:", orders.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)
print("payments:", payments.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)

orders: (99441, 11)
customers: (99441, 8)
order_items: (100000, 10)
payments: (100000, 8)
products: (32951, 12)
sellers: (3095, 7)


In [3]:
print("orders duplicates:", orders.duplicated().sum())
print("customers duplicates:", customers.duplicated().sum())
print("order_items duplicates:", order_items.duplicated().sum())
print("payments duplicates:", payments.duplicated().sum())
print("products duplicates:", products.duplicated().sum())
print("sellers duplicates:", sellers.duplicated().sum())

orders duplicates: 0
customers duplicates: 0
order_items duplicates: 0
payments duplicates: 0
products duplicates: 0
sellers duplicates: 0


In [4]:
# Convert date columns in orders
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"], errors="coerce")
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"], errors="coerce")
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"], errors="coerce")
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"], errors="coerce")

print("Orders datetime conversion done")

Orders datetime conversion done


In [5]:
print("order_purchase_timestamp nulls:", orders["order_purchase_timestamp"].isna().sum())
print("order_approved_at nulls:", orders["order_approved_at"].isna().sum())
print("order_delivered_carrier_date nulls:", orders["order_delivered_carrier_date"].isna().sum())
print("order_delivered_customer_date nulls:", orders["order_delivered_customer_date"].isna().sum())
print("order_estimated_delivery_date nulls:", orders["order_estimated_delivery_date"].isna().sum())

order_purchase_timestamp nulls: 0
order_approved_at nulls: 160
order_delivered_carrier_date nulls: 1783
order_delivered_customer_date nulls: 2965
order_estimated_delivery_date nulls: 0


In [6]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders[["order_purchase_timestamp", "order_delivered_customer_date", "delivery_days"]].head(5)

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


In [7]:
orders["days_delivery_vs_estimate"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders[["order_delivered_customer_date", "order_estimated_delivery_date", "days_delivery_vs_estimate"]].head(5)

,order_delivered_customer_date,order_estimated_delivery_date,days_delivery_vs_estimate
0,2017-10-10 21:25:13,2017-10-18,-8.0
1,2018-08-07 15:27:45,2018-08-13,-6.0
2,2018-08-17 18:06:29,2018-09-04,-18.0
3,2017-12-02 00:28:42,2017-12-15,-13.0
4,2018-02-16 18:17:02,2018-02-26,-10.0


In [8]:
orders["is_late_delivery"] = orders["days_delivery_vs_estimate"].apply(
    lambda x: 1 if x > 0 else 0
)

orders[[
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "days_delivery_vs_estimate",
    "is_late_delivery"
]].head(5)

,order_delivered_customer_date,order_estimated_delivery_date,days_delivery_vs_estimate,is_late_delivery
0,2017-10-10 21:25:13,2017-10-18,-8.0,0
1,2018-08-07 15:27:45,2018-08-13,-6.0,0
2,2018-08-17 18:06:29,2018-09-04,-18.0,0
3,2017-12-02 00:28:42,2017-12-15,-13.0,0
4,2018-02-16 18:17:02,2018-02-26,-10.0,0


In [9]:
orders_customers = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

print("Joined shape:", orders_customers.shape)
orders_customers.head(2)

Joined shape: (99441, 21)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at_x,_source_endpoint_x,...,delivery_days,days_delivery_vs_estimate,is_late_delivery,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_ingested_at_y,_source_endpoint_y,_page_number_y
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2026-03-26 20:13:53,orders,...,8.0,-8.0,0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,2026-03-26 20:22:20,customers,71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2026-03-26 20:13:53,orders,...,13.0,-6.0,0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,2026-03-26 20:22:29,customers,78


In [10]:
print("Orders rows:", len(orders))
print("Orders + Customers rows:", len(orders_customers))

Orders rows: 99441
Orders + Customers rows: 99441


In [11]:
orders_items = orders.merge(
    order_items,
    on="order_id",
    how="left"
)

print("Orders rows:", len(orders))
print("Orders + Items rows:", len(orders_items))
orders_items.head(2)

Orders rows: 99441
Orders + Items rows: 111868


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at_x,_source_endpoint_x,...,is_late_delivery,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_ingested_at_y,_source_endpoint_y,_page_number_y
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2026-03-26 20:13:53,orders,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2026-03-26 20:13:53,orders,...,0,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.7,22.76,2026-03-26 20:27:05,order_items,37.0


In [13]:
orders_filtered = orders[orders["order_purchase_timestamp"] >= "2018-07-01"].copy()

print("Original orders rows:", len(orders))
print("Filtered orders rows:", len(orders_filtered))
orders_filtered[["order_id", "order_purchase_timestamp"]].head(5)

Original orders rows: 99441
Filtered orders rows: 12824


,order_id,order_purchase_timestamp
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49
13,5ff96c15d0b717ac6ad1f3d77225a350,2018-07-25 17:44:10
24,f3e7c359154d965827355f39d6b1fdac,2018-08-09 11:44:40
34,b276e4f8c0fb86bd82fce576f21713e0,2018-07-29 23:34:51


In [14]:
orders_items_filtered = orders_filtered.merge(
    order_items,
    on="order_id",
    how="left"
)

print("Filtered orders rows:", len(orders_filtered))
print("After join rows:", len(orders_items_filtered))
orders_items_filtered.head(2)

Filtered orders rows: 12824
After join rows: 14273


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at_x,_source_endpoint_x,...,is_late_delivery,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_ingested_at_y,_source_endpoint_y,_page_number_y
0,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2026-03-26 20:13:53,orders,...,0,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.7,22.76,2026-03-26 20:27:05,order_items,37.0
1,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2026-03-26 20:13:53,orders,...,0,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.9,19.22,2026-03-26 20:27:02,order_items,32.0


In [15]:
payments_grouped = payments.groupby("order_id", as_index=False).agg({
    "payment_value": "sum",
    "payment_installments": "max"
})

print("Grouped payments rows:", len(payments_grouped))
payments_grouped.head(5)

Grouped payments rows: 95839


,order_id,payment_value,payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,3
2,000229ec398224ef6ca0657da4fc703e,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3


In [16]:
payment_type_df = payments.sort_values(
    by=["order_id", "payment_value", "payment_type"],
    ascending=[True, False, True]
).drop_duplicates(subset=["order_id"])[["order_id", "payment_type"]]

print("Payment type rows:", len(payment_type_df))
payment_type_df.head(5)

Payment type rows: 95839


,order_id,payment_type
85283,00010242fe8c5a6d1ba2dd792cb16214,credit_card
2499,00018f77f2f0320c557190d7a144bdd3,credit_card
12393,000229ec398224ef6ca0657da4fc703e,credit_card
32971,00024acbcdf0a6daa1e931b038114c75,credit_card
98711,00042b26cf59d7ce69dfabb4e55b4fd9,credit_card


In [17]:
payments_final = payments_grouped.merge(
    payment_type_df,
    on="order_id",
    how="left"
)

print("payments_final rows:", len(payments_final))
payments_final.head(5)

payments_final rows: 95839


,order_id,payment_value,payment_installments,payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card


In [18]:
fact_base = orders_items_filtered.merge(
    payments_final,
    on="order_id",
    how="left"
)

print("fact_base rows:", len(fact_base))
fact_base.head(2)

fact_base rows: 14273


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at_x,_source_endpoint_x,...,seller_id,shipping_limit_date,price,freight_value,_ingested_at_y,_source_endpoint_y,_page_number_y,payment_value,payment_installments,payment_type
0,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2026-03-26 20:13:53,orders,...,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.7,22.76,2026-03-26 20:27:05,order_items,37.0,141.46,1.0,boleto
1,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2026-03-26 20:13:53,orders,...,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.9,19.22,2026-03-26 20:27:02,order_items,32.0,179.12,3.0,credit_card


In [19]:
# Total item price per order
order_price_totals = fact_base.groupby("order_id", as_index=False)["price"].sum()
order_price_totals.rename(columns={"price": "order_total_price"}, inplace=True)

# Join total price back
fact_base = fact_base.merge(order_price_totals, on="order_id", how="left")

# Distribute payment_value proportionally
fact_base["payment_value_distributed"] = (
    (fact_base["price"] / fact_base["order_total_price"]) * fact_base["payment_value"]
)

fact_base[[
    "order_id",
    "order_item_id",
    "price",
    "order_total_price",
    "payment_value",
    "payment_value_distributed"
]].head(10)

,order_id,order_item_id,price,order_total_price,payment_value,payment_value_distributed
0,53cdb2fc8bc7dce0b6741e2150273451,1.0,118.7,118.7,141.46,141.46
1,47770eb9100c2d0c44946d9cf07ec65d,1.0,159.9,159.9,179.12,179.12
2,5ff96c15d0b717ac6ad1f3d77225a350,1.0,19.9,19.9,32.70,32.70
3,f3e7c359154d965827355f39d6b1fdac,NaN,NaN,0.0,104.11,NaN
4,b276e4f8c0fb86bd82fce576f21713e0,1.0,179.0,179.0,188.41,188.41
5,d22e9fa5731b9e30e8b27afcdc2f8563,1.0,99.0,99.0,121.62,121.62
6,5820a1100976432c7968a52da59e9364,1.0,33.9,33.9,52.24,52.24
7,9faeb9b2746b9d7526aef5acb08e2aa0,1.0,60.0,120.0,151.04,75.52
8,9faeb9b2746b9d7526aef5acb08e2aa0,2.0,60.0,120.0,151.04,75.52
9,f346ad4ee8f630e5e4ddaf862a34e6dd,NaN,NaN,0.0,NaN,NaN


In [20]:
fact_base["_is_valid"] = (
    fact_base["order_item_id"].notna() &
    fact_base["price"].notna() &
    fact_base["product_id"].notna() &
    fact_base["seller_id"].notna()
)

print(fact_base["_is_valid"].value_counts(dropna=False))

_is_valid
True     12768
False     1505
Name: count, dtype: int64


In [21]:
fact_valid = fact_base[fact_base["_is_valid"] == True].copy()

print("fact_base rows:", len(fact_base))
print("fact_valid rows:", len(fact_valid))
fact_valid.head(2)

fact_base rows: 14273
fact_valid rows: 12768


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at_x,_source_endpoint_x,...,freight_value,_ingested_at_y,_source_endpoint_y,_page_number_y,payment_value,payment_installments,payment_type,order_total_price,payment_value_distributed,_is_valid
0,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2026-03-26 20:13:53,orders,...,22.76,2026-03-26 20:27:05,order_items,37.0,141.46,1.0,boleto,118.7,141.46,True
1,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2026-03-26 20:13:53,orders,...,19.22,2026-03-26 20:27:02,order_items,32.0,179.12,3.0,credit_card,159.9,179.12,True


In [22]:
fact_order_items = fact_valid[[
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "customer_id",
    "order_purchase_timestamp",
    "price",
    "freight_value",
    "payment_value_distributed",
    "payment_type",
    "payment_installments",
    "delivery_days",
    "is_late_delivery"
]].copy()

print("Final fact shape:", fact_order_items.shape)
fact_order_items.head(2)

Final fact shape: (12768, 13)


,order_id,order_item_id,product_id,seller_id,customer_id,order_purchase_timestamp,price,freight_value,payment_value_distributed,payment_type,payment_installments,delivery_days,is_late_delivery
0,53cdb2fc8bc7dce0b6741e2150273451,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,118.7,22.76,141.46,boleto,1.0,13.0,0
1,47770eb9100c2d0c44946d9cf07ec65d,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,159.9,19.22,179.12,credit_card,3.0,9.0,0


In [23]:
dim_customers = customers[[
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]].drop_duplicates().copy()

print("dim_customers shape:", dim_customers.shape)
dim_customers.head(5)

dim_customers shape: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [24]:
dim_customers = dim_customers.drop_duplicates(subset=["customer_unique_id"])

print("Final dim_customers shape:", dim_customers.shape)

Final dim_customers shape: (96096, 5)


In [25]:
dim_products = products[[
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]].copy()

print("dim_products shape:", dim_products.shape)
dim_products.head(5)

dim_products shape: (32951, 6)


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0


In [26]:
dim_sellers = sellers[[
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
]].drop_duplicates().copy()

print("dim_sellers shape:", dim_sellers.shape)
dim_sellers.head(5)

dim_sellers shape: (3095, 4)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [27]:
dim_products["product_category_name"] = dim_products["product_category_name"].fillna("unknown")

dim_products["product_volume_cm3"] = (
    dim_products["product_length_cm"] *
    dim_products["product_height_cm"] *
    dim_products["product_width_cm"]
)

dim_products[[
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_volume_cm3"
]].head(5)

,product_id,product_category_name,product_weight_g,product_volume_cm3
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,2240.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,10800.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,2430.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,2704.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,4420.0


In [28]:
fact_order_items.to_csv("/home/jovyan/work/output/gold/fact_order_items.csv", index=False)
dim_customers.to_csv("/home/jovyan/work/output/gold/dim_customers.csv", index=False)
dim_products.to_csv("/home/jovyan/work/output/gold/dim_products.csv", index=False)
dim_sellers.to_csv("/home/jovyan/work/output/gold/dim_sellers.csv", index=False)

print("All gold files saved successfully")

All gold files saved successfully


In [29]:
import os

os.makedirs("/home/jovyan/work/output/silver", exist_ok=True)

orders_filtered.to_csv("/home/jovyan/work/output/silver/orders_filtered.csv", index=False)
fact_valid.to_csv("/home/jovyan/work/output/silver/fact_base_clean.csv", index=False)

print("silver files saved")

silver files saved


In [10]:
import os

print(os.listdir("/home/jovyan/Submission/output"))

['bronze', 'silver', '.gitkeep', '.ipynb_checkpoints', 'gold']


In [11]:
import os

print(os.listdir("/home/jovyan/Submission/output"))

['bronze', 'silver', '.gitkeep', '.ipynb_checkpoints', 'gold']


In [13]:
fact_valid = pd.read_csv("/home/jovyan/Submission/output/silver/fact_base_clean.csv")

In [15]:
import pandas as pd

BASE = "/home/jovyan/Submission/output"

# silver
orders_filtered = pd.read_csv(f"{BASE}/silver/orders_filtered.csv")
fact_valid = pd.read_csv(f"{BASE}/silver/fact_base_clean.csv")

# bronze
order_items = pd.read_csv(f"{BASE}/bronze/order_items.csv")
customers = pd.read_csv(f"{BASE}/bronze/customers.csv")
products = pd.read_csv(f"{BASE}/bronze/products.csv")
sellers = pd.read_csv(f"{BASE}/bronze/sellers.csv")

print("✅ all data loaded")



✅ all data loaded


In [16]:

print(orders_filtered.shape, order_items.shape, customers.shape)

(12824, 14) (100000, 10) (99441, 8)


In [18]:
cust_orders = orders_filtered.merge(order_items, on="order_id", how="left") \
                             .merge(customers, on="customer_id", how="left")

In [19]:
import hashlib

def stable_int_key(value: str):
    if pd.isna(value):
        return None
    return int(hashlib.md5(str(value).encode()).hexdigest()[:8], 16)

cust_orders = orders_filtered.merge(order_items, on="order_id", how="left") \
                             .merge(customers, on="customer_id", how="left")

cust_agg = cust_orders.groupby("customer_unique_id", as_index=False).agg(
    first_order_date=("order_purchase_timestamp", "min"),
    total_orders=("order_id", "nunique"),
    total_spend=("price", "sum")
)

cust_agg["total_spend"] = cust_agg["total_spend"].fillna(0)
cust_agg["is_repeat_customer"] = cust_agg["total_orders"] > 1

cust_recent = cust_orders.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"],
    ascending=[True, False]
).drop_duplicates(subset=["customer_unique_id"])[
    ["customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"]
]

dim_customers_final = cust_recent.merge(cust_agg, on="customer_unique_id", how="left")
dim_customers_final["customer_key"] = dim_customers_final["customer_unique_id"].apply(stable_int_key)

print(dim_customers_final.shape)
dim_customers_final.head(2)

(12647, 9)


,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,first_order_date,total_orders,total_spend,is_repeat_customer,customer_key
0,000ec5bff359e1c0ad76a81a45cb598f,salto de pirapora,SP,18160,2018-08-21 11:34:26,1,14.96,False,1476504083
1,000fbf0473c10fc1ab6f8d2d286ce20c,indaiatuba,SP,13330,2018-07-26 09:43:52,1,285.80,False,214807387


In [20]:
import hashlib

def stable_int_key(value: str):
    if pd.isna(value):
        return None
    return int(hashlib.md5(str(value).encode()).hexdigest()[:8], 16)

cust_orders = orders_filtered.merge(order_items, on="order_id", how="left") \
                             .merge(customers, on="customer_id", how="left")

cust_agg = cust_orders.groupby("customer_unique_id", as_index=False).agg(
    first_order_date=("order_purchase_timestamp", "min"),
    total_orders=("order_id", "nunique"),
    total_spend=("price", "sum")
)

cust_agg["total_spend"] = cust_agg["total_spend"].fillna(0)
cust_agg["is_repeat_customer"] = cust_agg["total_orders"] > 1

cust_recent = cust_orders.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"],
    ascending=[True, False]
).drop_duplicates(subset=["customer_unique_id"])[
    ["customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"]
]

dim_customers_final = cust_recent.merge(cust_agg, on="customer_unique_id", how="left")
dim_customers_final["customer_key"] = dim_customers_final["customer_unique_id"].apply(stable_int_key)

print(dim_customers_final.shape)
dim_customers_final.head(2)

(12647, 9)


,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,first_order_date,total_orders,total_spend,is_repeat_customer,customer_key
0,000ec5bff359e1c0ad76a81a45cb598f,salto de pirapora,SP,18160,2018-08-21 11:34:26,1,14.96,False,1476504083
1,000fbf0473c10fc1ab6f8d2d286ce20c,indaiatuba,SP,13330,2018-07-26 09:43:52,1,285.80,False,214807387


In [21]:
dim_products_final = products.copy()

dim_products_final["product_category_name"] = dim_products_final["product_category_name"].fillna("unknown")

dim_products_final["product_volume_cm3"] = (
    dim_products_final["product_length_cm"] *
    dim_products_final["product_height_cm"] *
    dim_products_final["product_width_cm"]
)

dim_products_final["product_key"] = dim_products_final["product_id"].apply(stable_int_key)

# typo-safe rename if needed
if "product_description_lenght" in dim_products_final.columns:
    dim_products_final = dim_products_final.rename(
        columns={"product_description_lenght": "product_description_length"}
    )

dim_products_final = dim_products_final[[
    "product_key",
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_volume_cm3",
    "product_photos_qty",
    "product_description_length"
]].copy()

print(dim_products_final.shape)
dim_products_final.head(2)

(32951, 7)


,product_key,product_id,product_category_name,product_weight_g,product_volume_cm3,product_photos_qty,product_description_length
0,4141291038,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,2240.0,1.0,287.0
1,1924322346,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,10800.0,1.0,276.0


In [22]:
dim_products_final["product_key"] = dim_products_final["product_id"].apply(stable_int_key)

In [23]:
dim_products_final = products.copy()

dim_products_final["product_category_name"] = dim_products_final["product_category_name"].fillna("unknown")

dim_products_final["product_volume_cm3"] = (
    dim_products_final["product_length_cm"] *
    dim_products_final["product_height_cm"] *
    dim_products_final["product_width_cm"]
)

dim_products_final["product_key"] = dim_products_final["product_id"].apply(stable_int_key).astype("Int64")

if "product_description_lenght" in dim_products_final.columns:
    dim_products_final = dim_products_final.rename(
        columns={"product_description_lenght": "product_description_length"}
    )

dim_products_final = dim_products_final[[
    "product_key",
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_volume_cm3",
    "product_photos_qty",
    "product_description_length"
]].copy()

print(dim_products_final["product_key"].isna().sum())
print(dim_products_final.shape)
dim_products_final.head(2)

0
(32951, 7)


,product_key,product_id,product_category_name,product_weight_g,product_volume_cm3,product_photos_qty,product_description_length
0,4141291038,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,2240.0,1.0,287.0
1,1924322346,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,10800.0,1.0,276.0


In [24]:
dim_products_final = products.copy()

dim_products_final["product_category_name"] = dim_products_final["product_category_name"].fillna("unknown")

dim_products_final["product_volume_cm3"] = (
    dim_products_final["product_length_cm"] *
    dim_products_final["product_height_cm"] *
    dim_products_final["product_width_cm"]
)

dim_products_final["product_key"] = dim_products_final["product_id"].apply(stable_int_key).astype("Int64")

if "product_description_lenght" in dim_products_final.columns:
    dim_products_final = dim_products_final.rename(
        columns={"product_description_lenght": "product_description_length"}
    )

dim_products_final = dim_products_final[[
    "product_key",
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_volume_cm3",
    "product_photos_qty",
    "product_description_length"
]].copy()

print(dim_products_final["product_key"].isna().sum())
print(dim_products_final.shape)
dim_products_final.head(2)

0
(32951, 7)


,product_key,product_id,product_category_name,product_weight_g,product_volume_cm3,product_photos_qty,product_description_length
0,4141291038,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,2240.0,1.0,287.0
1,1924322346,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,10800.0,1.0,276.0


In [25]:
dim_sellers_final = sellers.copy()

dim_sellers_final["seller_key"] = dim_sellers_final["seller_id"].apply(stable_int_key).astype("Int64")

dim_sellers_final = dim_sellers_final[[
    "seller_key",
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix"
]].drop_duplicates().copy()

print(dim_sellers_final["seller_key"].isna().sum())
print(dim_sellers_final.shape)
dim_sellers_final.head(2)

0
(3095, 5)


,seller_key,seller_id,seller_city,seller_state,seller_zip_code_prefix
0,1421897088,3442f8959a84dea7ee197c632cb2df15,campinas,SP,13023
1,1040624557,d1b65fc7debc3361ea86b5f14c68d2e2,mogi guacu,SP,13844


In [26]:
fact_order_items_final = fact_valid.copy()

# customer_unique_id lao
fact_order_items_final = fact_order_items_final.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

# surrogate keys
fact_order_items_final["order_item_sk"] = (
    fact_order_items_final["order_id"].astype(str) + "_" +
    fact_order_items_final["order_item_id"].astype(str)
).apply(stable_int_key).astype("Int64")

fact_order_items_final["customer_key"] = fact_order_items_final["customer_unique_id"].apply(stable_int_key).astype("Int64")
fact_order_items_final["product_key"] = fact_order_items_final["product_id"].apply(stable_int_key).astype("Int64")
fact_order_items_final["seller_key"] = fact_order_items_final["seller_id"].apply(stable_int_key).astype("Int64")

# order_date
fact_order_items_final["order_date"] = pd.to_datetime(
    fact_order_items_final["order_purchase_timestamp"],
    errors="coerce"
).dt.date

# rename metrics
fact_order_items_final["days_to_deliver"] = fact_order_items_final["delivery_days"]
fact_order_items_final["payment_value"] = fact_order_items_final["payment_value_distributed"]

# final columns (requirement based)
fact_order_items_final = fact_order_items_final[[
    "order_item_sk",
    "order_id",
    "order_item_id",
    "customer_key",
    "product_key",
    "seller_key",
    "order_date",
    "order_status",
    "price",
    "freight_value",
    "payment_value",
    "payment_type",
    "payment_installments",
    "days_to_deliver",
    "days_delivery_vs_estimate",
    "is_late_delivery"
]].copy()

print(fact_order_items_final.shape)
fact_order_items_final.head(2)

(12768, 16)


,order_item_sk,order_id,order_item_id,customer_key,product_key,seller_key,order_date,order_status,price,freight_value,payment_value,payment_type,payment_installments,days_to_deliver,days_delivery_vs_estimate,is_late_delivery
0,1410316122,53cdb2fc8bc7dce0b6741e2150273451,1.0,3419462141,2746034656,261066546,2018-07-24,delivered,118.7,22.76,141.46,boleto,1.0,13.0,-6.0,0
1,776082596,47770eb9100c2d0c44946d9cf07ec65d,1.0,3328085264,3830244002,2622583364,2018-08-08,delivered,159.9,19.22,179.12,credit_card,3.0,9.0,-18.0,0


In [27]:
print(fact_order_items_final.isna().sum())

order_item_sk                  0
order_id                       0
order_item_id                  0
customer_key                   0
product_key                    0
seller_key                     0
order_date                     0
order_status                   0
price                          0
freight_value                  0
payment_value                444
payment_type                 444
payment_installments         444
days_to_deliver              211
days_delivery_vs_estimate    211
is_late_delivery               0
dtype: int64


In [28]:
# payment null handling
fact_order_items_final["payment_value"] = fact_order_items_final["payment_value"].fillna(0)
fact_order_items_final["payment_installments"] = fact_order_items_final["payment_installments"].fillna(0)
fact_order_items_final["payment_type"] = fact_order_items_final["payment_type"].fillna("unknown")

# is_late_delivery should be NULL if not delivered
fact_order_items_final["is_late_delivery"] = fact_order_items_final.apply(
    lambda row: None if pd.isna(row["days_delivery_vs_estimate"])
    else (1 if row["days_delivery_vs_estimate"] > 0 else 0),
    axis=1
)

print(fact_order_items_final.isna().sum())
fact_order_items_final.head(2)

order_item_sk                  0
order_id                       0
order_item_id                  0
customer_key                   0
product_key                    0
seller_key                     0
order_date                     0
order_status                   0
price                          0
freight_value                  0
payment_value                  0
payment_type                   0
payment_installments           0
days_to_deliver              211
days_delivery_vs_estimate    211
is_late_delivery             211
dtype: int64


,order_item_sk,order_id,order_item_id,customer_key,product_key,seller_key,order_date,order_status,price,freight_value,payment_value,payment_type,payment_installments,days_to_deliver,days_delivery_vs_estimate,is_late_delivery
0,1410316122,53cdb2fc8bc7dce0b6741e2150273451,1.0,3419462141,2746034656,261066546,2018-07-24,delivered,118.7,22.76,141.46,boleto,1.0,13.0,-6.0,0.0
1,776082596,47770eb9100c2d0c44946d9cf07ec65d,1.0,3328085264,3830244002,2622583364,2018-08-08,delivered,159.9,19.22,179.12,credit_card,3.0,9.0,-18.0,0.0


In [29]:
print("fact rows:", len(fact_order_items_final))
print("dim_customers rows:", len(dim_customers_final))
print("dim_products rows:", len(dim_products_final))
print("dim_sellers rows:", len(dim_sellers_final))

print("missing customer keys:",
      (~fact_order_items_final["customer_key"].isin(dim_customers_final["customer_key"])).sum())

print("missing product keys:",
      (~fact_order_items_final["product_key"].isin(dim_products_final["product_key"])).sum())

print("missing seller keys:",
      (~fact_order_items_final["seller_key"].isin(dim_sellers_final["seller_key"])).sum())

fact rows: 12768
dim_customers rows: 12647
dim_products rows: 32951
dim_sellers rows: 3095
missing customer keys: 0
missing product keys: 0
missing seller keys: 0


In [30]:
import os

BASE_OUT = "/home/jovyan/Submission/output/gold"
os.makedirs(BASE_OUT, exist_ok=True)

fact_order_items_final.to_csv(f"{BASE_OUT}/fact_order_items.csv", index=False)
dim_customers_final.to_csv(f"{BASE_OUT}/dim_customers.csv", index=False)
dim_products_final.to_csv(f"{BASE_OUT}/dim_products.csv", index=False)
dim_sellers_final.to_csv(f"{BASE_OUT}/dim_sellers.csv", index=False)

print("gold final saved")

gold final saved
